# Análise de dados na prática: investigando o Titanic

**Ciclo que vamos seguir:** pergunta → dados → limpeza → exploração → resposta → comunicação.

Em cada etapa, escreva sua hipótese **antes** de rodar a célula.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = sns.load_dataset("titanic")
df.head()

## Parte 0: conhecendo os dados
**Pergunta:** o que tem nessa tabela? Cada linha é o quê?

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.describe(include="all").T

## Parte 1: quantos?
**Pergunta:** quantos passageiros temos e qual a proporção de sobreviventes?

In [ ]:
print("Passageiros:", len(df))
print("Taxa de sobrevivência:", round(df["survived"].mean() * 100, 1), "%")

## Parte 2: o que está faltando?
**Pergunta:** quais colunas têm valores nulos? Isso é um problema?

In [ ]:
nulos = df.isna().mean().sort_values(ascending=False) * 100
nulos[nulos > 0].round(1)

`deck` tem muitos nulos (quase sempre não foi registrado). `age` tem uns 20%.

**Decisão de analista:** o que fazer? Remover linhas? Preencher com a mediana? Ignorar a coluna? Cada escolha muda a conclusão, então registre qual você tomou e por quê.

In [ ]:
# Para a idade, vamos só ignorar os nulos nas análises que usarem essa coluna
# (sem inventar valores) e lembrar que a amostra ali é menor.
df["age"].describe()

## Parte 3: quem sobreviveu?
**Pergunta:** sexo e classe influenciaram a chance de sobreviver?

In [ ]:
print(df.groupby("sex")["survived"].mean().round(2))
print()
print(df.groupby("pclass")["survived"].mean().round(2))

In [ ]:
sns.barplot(data=df, x="pclass", y="survived", hue="sex")
plt.title("Taxa de sobrevivência por classe e sexo")
plt.ylabel("Taxa de sobrevivência")
plt.show()

## Parte 4: e a idade?
**Pergunta:** crianças tiveram mais chance? (Lembre dos nulos em `age`.)

In [ ]:
df["faixa_etaria"] = pd.cut(
    df["age"],
    bins=[0, 12, 18, 40, 60, 100],
    labels=["0-12", "13-18", "19-40", "41-60", "60+"],
)
print(df.groupby("faixa_etaria", observed=True)["survived"].agg(["mean", "count"]).round(2))

Repare na coluna `count`: faixas com poucos passageiros dão taxas instáveis.

## Parte 5: outliers
**Pergunta:** a tarifa (`fare`) tem valores estranhos?

In [ ]:
sns.boxplot(data=df, x="pclass", y="fare")
plt.show()
df.sort_values("fare", ascending=False)[["pclass", "sex", "age", "fare"]].head()

Tarifas muito altas na 1ª classe são **reais**, não erro. Já tarifa 0 pode ser dado faltante ou cortesia. Outlier não é sinônimo de "apagar".

In [ ]:
(df["fare"] == 0).sum()

## Parte 6: correlação não é causa
**Pergunta:** a 1ª classe sobreviveu mais *porque* era 1ª classe?

Veja: a composição por sexo é igual em todas as classes?

In [ ]:
print(pd.crosstab(df["pclass"], df["sex"], normalize="index").round(2))
print()
print(df.pivot_table(index="pclass", columns="sex", values="survived", aggfunc="mean").round(2))

Comparar dentro de cada grupo (como na tabela acima) ajuda a separar efeitos, mas **ainda não prova causa**. Fatores como localização da cabine e acesso aos botes não estão nos dados.

## Parte 7: uma pergunta que os dados não respondem
**Pergunta:** "Se o navio tivesse mais botes, quantos teriam sobrevivido?"

Discuta: o que faltaria para responder? Dados sobre botes, horários, posição das cabines... Saber dizer **"esses dados não respondem isso"** também é resultado de análise.

## Parte 8: e se quisermos prever?
**Pergunta:** com poucas colunas, conseguimos prever quem sobreviveria?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = df[["pclass", "sex", "age", "fare"]].copy()
X["sex"] = (X["sex"] == "female").astype(int)
X["age"] = X["age"].fillna(X["age"].median())
y = df["survived"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

# Baseline: prever que ninguém sobreviveu
print("Baseline:", round(1 - y_te.mean(), 2))

modelo = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
print("Árvore:", round(accuracy_score(y_te, modelo.predict(X_te)), 2))

Sempre compare com um **baseline**. Um modelo só é bom se supera o chute óbvio. Aqui entra a fronteira entre analista e cientista de dados.

## Parte 9: comunicando
Escreva **3 frases** para alguém sem conhecimento técnico:

1. O que encontramos?
2. Quão confiável é? (nulos, amostra, correlação)
3. O que NÃO conseguimos afirmar?

## Desafio final
Formule **sua própria pergunta** sobre este dataset (ex.: quem viajava sozinho sobreviveu mais? `alone`, `embark_town`) e responda usando o ciclo completo.